In [ ]:
import numpy as np
import pandas as pd

from datasets import load_dataset

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import accuracy_score, f1_score

from scipy.stats import pearsonr, spearmanr

from gensim.models import Word2Vec

from sentence_transformers import SentenceTransformer

import matplotlib.pyplot as plt

In [ ]:
dataset = load_dataset("stsb_multi_mt", "en")

train_data = dataset["train"]
test_data = dataset["test"]

train_df = pd.DataFrame({
    "sentence1": train_data["sentence1"],
    "sentence2": train_data["sentence2"],
    "score": train_data["similarity_score"]
})

test_df = pd.DataFrame({
    "sentence1": test_data["sentence1"],
    "sentence2": test_data["sentence2"],
    "score": test_data["similarity_score"]
})

print(train_df.head())
print()
print(f"Train size: {len(train_df)}")
print(f"Test size: {len(test_df)}")

In [ ]:
train_df["score"].hist(bins=20)

plt.xlabel("Similarity Score")
plt.ylabel("Count")
plt.title("STS-B Similarity Score Distribution")

plt.show()

In [ ]:
tfidf = TfidfVectorizer()

all_sentences = list(train_df["sentence1"]) + list(train_df["sentence2"])

tfidf.fit(all_sentences)

In [ ]:
def tfidf_similarity(s1, s2):
    v1 = tfidf.transform([s1])
    v2 = tfidf.transform([s2])

    return cosine_similarity(v1, v2)[0][0]

In [ ]:
tfidf_predictions = []

for _, row in test_df.iterrows():
    sim = tfidf_similarity(row["sentence1"], row["sentence2"])
    tfidf_predictions.append(sim)

pearson = pearsonr(test_df["score"], tfidf_predictions)[0]
spearman = spearmanr(test_df["score"], tfidf_predictions)[0]

print("TF-IDF Results")
print("Pearson:", pearson)
print("Spearman:", spearman)

In [ ]:
sentences = all_sentences

tokenized = [s.lower().split() for s in sentences]

w2v_model = Word2Vec(
    sentences=tokenized,
    vector_size=100,
    window=5,
    min_count=1,
    workers=4
)

In [ ]:
def sentence_embedding(sentence):
    words = sentence.lower().split()

    vectors = []

    for w in words:
        if w in w2v_model.wv:
            vectors.append(w2v_model.wv[w])

    if len(vectors) == 0:
        return np.zeros(100)

    return np.mean(vectors, axis=0)

In [ ]:
def w2v_similarity(s1, s2):
    v1 = sentence_embedding(s1)
    v2 = sentence_embedding(s2)

    return np.dot(v1, v2) / (
        np.linalg.norm(v1) * np.linalg.norm(v2) + 1e-8
    )

In [ ]:
w2v_predictions = []

for _, row in test_df.iterrows():
    sim = w2v_similarity(row["sentence1"], row["sentence2"])
    w2v_predictions.append(sim)

pearson = pearsonr(test_df["score"], w2v_predictions)[0]
spearman = spearmanr(test_df["score"], w2v_predictions)[0]

print("Word2Vec Results")
print("Pearson:", pearson)
print("Spearman:", spearman)

In [ ]:
sbert = SentenceTransformer("all-MiniLM-L12-v2")

In [ ]:
def sbert_similarity(s1, s2):
    embeddings = sbert.encode([s1, s2])

    v1 = embeddings[0]
    v2 = embeddings[1]

    return np.dot(v1, v2) / (
        np.linalg.norm(v1) * np.linalg.norm(v2) + 1e-8
    )

In [ ]:
sbert_predictions = []

for _, row in test_df.iterrows():
    sim = sbert_similarity(row["sentence1"], row["sentence2"])
    sbert_predictions.append(sim)

pearson = pearsonr(test_df["score"], sbert_predictions)[0]
spearman = spearmanr(test_df["score"], sbert_predictions)[0]

print("SBERT Results")
print("Pearson:", pearson)
print("Spearman:", spearman)

In [ ]:
results = pd.DataFrame({
    "Model": ["TF-IDF", "Word2Vec", "SBERT"],
    "Pearson": [
        pearsonr(test_df["score"], tfidf_predictions)[0],
        pearsonr(test_df["score"], w2v_predictions)[0],
        pearsonr(test_df["score"], sbert_predictions)[0]
    ],
    "Spearman": [
        spearmanr(test_df["score"], tfidf_predictions)[0],
        spearmanr(test_df["score"], w2v_predictions)[0],
        spearmanr(test_df["score"], sbert_predictions)[0]
    ]
})

results

In [ ]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

In [ ]:
from sentence_transformers import SentenceTransformer, InputExample, losses
from torch.utils.data import DataLoader

model = SentenceTransformer("all-MiniLM-L12-v2")

train_samples = []

for _, row in train_df.sample(5000).iterrows():
    train_samples.append(
        InputExample(
            texts=[row["sentence1"], row["sentence2"]],
            label=float(row["score"]) / 5.0  # normalize 0–1
        )
    )

train_loader = DataLoader(train_samples, shuffle=True, batch_size=16)

loss = losses.CosineSimilarityLoss(model)

model.fit(
    train_objectives=[(train_loader, loss)],
    epochs=3,
    warmup_steps=100
)

In [ ]:
fine_tuned_sbert = model
def ft_sbert_similarity(s1, s2):
    emb = fine_tuned_sbert.encode([s1, s2])
    v1, v2 = emb[0], emb[1]

    return np.dot(v1, v2) / (
        np.linalg.norm(v1) * np.linalg.norm(v2) + 1e-8
    )

In [ ]:
def compare_sentences(s1, s2):
    print("\nSentence 1:", s1)
    print("Sentence 2:", s2)
    print("-" * 50)

    tfidf_score = tfidf_similarity(s1, s2)
    w2v_score = w2v_similarity(s1, s2)
    sbert_score = sbert_similarity(s1, s2)
    ft_sbert = ft_sbert_similarity(s1, s2)

    print(f"TF-IDF similarity : {tfidf_score:.4f}")
    print(f"Word2Vec similarity: {w2v_score:.4f}")
    print(f"SBERT similarity   : {sbert_score:.4f}")
    print(f"Fine tuned SBERT similarity   : {ft_sbert:.4f}")

In [ ]:
compare_sentences(
    "The cat is sleeping on the sofa.",
    "The feline is dozing on the couch."
)

In [ ]:
compare_sentences(
    "The chicken is ready to eat.",
    "The chicken is eating."
)

compare_sentences(
    "I saw the man with the telescope.",
    "I used a telescope to see the man."
)

